# Benchmark review notebook - ENBOLSA

Notebook corto de revision visual para la comparativa entre ENBOLSA y benchmarks clasicos.

Objetivos:
- revisar tablas canonicas clave;
- visualizar los graficos mas representativos;
- disponer de un apoyo rapido antes de integrar esta seccion en la memoria del TFG.

In [ ]:
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

candidates = [
    Path.cwd(),
    Path.cwd() / 'artifacts' / 'benchmark-significance' / 'enbolsa' / 'final',
]
ROOT = next((p for p in candidates if (p / 'tables').exists() and (p / 'charts').exists()), None)
if ROOT is None:
    raise FileNotFoundError('No se encontro la carpeta final/ con tables y charts.')

TABLES = ROOT / 'tables'
CHARTS = ROOT / 'charts'

aggregate_by_strategy = pd.read_csv(TABLES / 'aggregate_by_strategy.csv')
aggregate_by_group = pd.read_csv(TABLES / 'aggregate_by_group.csv')
aggregate_by_tf_pair = pd.read_csv(TABLES / 'aggregate_by_tf_pair.csv')
block_metrics = pd.read_csv(TABLES / 'block_metrics.csv')
trade_pool_global = pd.read_csv(TABLES / 'trade_pool_global.csv')
sp500_context = pd.read_csv(TABLES / 'sp500_buy_hold_context.csv')

display(Markdown(f'**Ruta canonica cargada:** `{ROOT}`'))

In [ ]:
display(Markdown('## Resumen global por estrategia'))
cols = [
    'Variante', 'Blocks', 'TotalTrades', 'TotalNetProfit', 'MeanReturn%',
    'MedianReturn%', 'PositiveBlockRate%', 'MedianPF', 'MedianAvgR'
]
display(aggregate_by_strategy[cols].sort_values('MeanReturn%', ascending=False))

display(Markdown('## Resumen por grupo'))
group_cols = [
    'Variante', 'Group', 'Blocks', 'TotalTrades', 'TotalNetProfit',
    'MeanReturn%', 'MedianReturn%', 'PositiveBlockRate%', 'MedianPF'
]
display(aggregate_by_group[group_cols].sort_values(['Group', 'MeanReturn%'], ascending=[True, False]))

display(Markdown('## Resumen por pareja temporal'))
tf_cols = [
    'Variante', 'TFPair', 'Blocks', 'TotalTrades', 'TotalNetProfit',
    'MeanReturn%', 'MedianReturn%', 'PositiveBlockRate%', 'MedianPF'
]
display(aggregate_by_tf_pair[tf_cols].sort_values(['TFPair', 'MeanReturn%'], ascending=[True, False]))

In [ ]:
display(Markdown('## Metrica canonica por bloque'))
block_cols = [
    'Variante', 'Group', 'TFPair', 'BlockId', 'Trades', 'Return%',
    'PF', 'MaxDD%', 'Sharpe', 'NetProfit', 'TFStackEffective', 'H4D1Mode'
]
display(block_metrics[block_cols].sort_values(['Group', 'TFPair', 'Variante']))

display(Markdown('## Trade pool global'))
display(trade_pool_global[['Variante', 'Trades', 'WR%', 'PF', 'NetProfit', 'Expectancy', 'AvgR', 'ExpectancyR']].sort_values('NetProfit', ascending=False))

display(Markdown('## Contexto buy and hold sobre US500'))
display(sp500_context)

In [ ]:
chart_names = [
    'heatmap_returnpct_por_bloque.png',
    'lineas_r_acumulada_por_trade.png',
    'histograma_densidad_returnpct_por_bloque.png',
    'heatmap_trade_pool_netprofit_activo_estrategia.png',
]

for chart_name in chart_names:
    path = CHARTS / chart_name
    if not path.exists():
        display(Markdown(f'**No encontrado:** `{chart_name}`'))
        continue
    img = mpimg.imread(path)
    plt.figure(figsize=(14, 7))
    plt.imshow(img)
    plt.axis('off')
    plt.title(chart_name)
    plt.show()

## Lectura rapida

- `enbolsa:macd_breakout` es la unica estrategia con ventaja global clara frente a los benchmarks clasicos.
- `enbolsa:fib_limit` queda en una posicion intermedia: no lidera el conjunto, pero supera a la mayoria de benchmarks simples.
- `Metals` es el grupo donde los benchmarks muestran un comportamiento relativamente mas competitivo, especialmente `ma_cross_3tf_trend`.
- `H4:D1` debe interpretarse como variante degradada 2TF, no como stack 3TF completo.
- La curva de lineas refleja R acumulada por numero de trade en el trade-pool, no una equity unica de cartera.
- La comparacion mantiene entrada a cierre para preservar comparabilidad con ENBOLSA, lo que introduce una hipotesis de backtest ligeramente optimista frente a una ejecucion estricta en la vela siguiente.